# Web scraping - Sites dinâmicos

Na aula de hoje veremos:
 - Como encontrar elementos em uma página web
 - Como navegar em diferentes páginas
 - Como preencher formulários
 - Como lidar com estratégias anti-scraping
 - O que são servidores proxies e como utilizá-los
 - Como bloquear requisições

### Encontrando elementos 


| ABORDAGEM       | DESCRIÇÃO                                                       | HTML                                                            | SELENIUM                                                                                                           |
|:----------------|:----------------------------------------------------------------|:----------------------------------------------------------------|:------------------------------------------------------------------------------------------------------------------|
| By.ID           | Seleciona elemento HTML com base no id attribute                | \<div id="s-437">...\</div>                                      | find_element(By.ID, "s-437")                                                                                      |
| By.NAME         | Seleciona elemento HTML com base no name attribute              | \<input name="email" />                                         | find_element(By.NAME, "email") <br> find_elements(By.NAME, "email")                                               |
| By.XPATH        | Seleciona elemento HTML que dá match no XPath expression        | \<h1>My <strong>Fantastic</strong> Blog\</h1>                    | find_element(By.XPATH, "//h1/strong") <br> find_elements(By.XPATH, "//h1/strong")                                 |
| By.LINK_TEXT    | Seleciona elemento \<a> HTML que contém o texto do link         | \<a href="/">Home\</a>                                           | find_element(By.LINK_TEXT, "Home") <br> find_elements(By.LINK_TEXT, "Home")                                       |
| By.TAG_NAME     | Seleciona elemento HTML com base no tag name                    | \<span>...\</span>                                               | find_element(By.TAG_NAME, "span") <br> find_elements(By.TAG_NAME, "span")                                         |
| By.CLASS_NAME   | Seleciona elemento HTML com base na class attribute             | \<div class="text-center"><br> Welcome! <br>    \</div>                        | find_element(By.CLASSNAME, "text-center") <br> find_elements(By.CLASSNAME, "text-center")                         |
| By.CSS_SELECTOR | Seleciona elemento HTML que dá match a CSS selector             | \<div class="product-card"> <br>       \<span class="price"\> </br> $140 </br> \</span> <br> \</div>| find_element(By.CSS_SELECTOR, ".product-card .price") <br> find_elements(By.CSS_SELECTOR, ".product-card .price")|


`find_element`: retorna o primeiro elemento que casa com o padrão buscado<br>
`find_elements`: retorna todos os elementos que casam com o padrão buscado     

### Lab

In [2]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

service = Service(ChromeDriverManager().install())

options = Options()
#options.headlessess = True
driver = webdriver.Chrome(service=service, options=options)
driver.get('https://www.scrapingcourse.com')
time.sleep(5)
driver.quit()

#### Find elements

In [ ]:
cards = driver.find_elements(By.CLASS_NAME, 'card card_container')
cards = driver.find_elements(By.CSS_SELECTOR, '.card.card_container')
cards = driver.find_elements(By.XPATH, '//*[@id="content-container"]/div/main/div/div[1]')
cards = driver.find_elements(By.TAG_NAME, 'a')

/html/body/main/div/div/main/div/div[1]

#### Find element

In [ ]:
card = driver.find_element(By.CLASS_NAME, 'card card_container')
card = driver.find_element(By.CSS_SELECTOR, '.card.card_container')
card = driver.find_element(By.XPATH, '//*[@id="content-container"]/div/main/div/div[1]')


### Opções

In [ ]:
opts = webdriver.ChromeOptions()

opts.add_argument('--headless=new')
opts.add_argument("--start-maximized")  # abre tela cheia
opts.add_argument("--window-size=1280,800")  # define tamanho manual
opts.add_argument("--incognito")  # modo anônimo
opts.add_argument("--disable-notifications")  # bloqueia pop-ups de notificação
opts.add_argument("--disable-extensions")  # desativa extensões
opts.add_argument("--disable-popup-blocking")  # desativa bloqueador de pop-ups
opts.add_argument("--disable-infobars")  # remove "Chrome is being controlled by automated test software"
opts.add_argument("--no-sandbox")  # útil em servidores Linux
opts.add_argument("--disable-dev-shm-usage")  # previne erros de memória em containers
opts.add_argument("--remote-debugging-port=9222")  # habilita inspeção remota

### Ecommerce (lista de produtos com paginação e preço)

In [4]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

service = Service(ChromeDriverManager().install())
options = Options()
#options.headlessess = True
driver = webdriver.Chrome(service=service, options=options)
driver.get('https://www.scrapingcourse.com')

# busca todos os links com o texto 'See Page'
links = driver.find_elements(By.LINK_TEXT, 'See Page')

if links:
    links[0].click()
    print(f'Navegou para: {driver.current_url}')
else:
    print('Nenhum link que atenda ao padrão buscado')

time.sleep(10)

driver.quit()

Navegou para: https://www.scrapingcourse.com/ecommerce/


### Pagination (lista paginada por números)

In [20]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

service = Service(ChromeDriverManager().install())

driver = webdriver.Chrome(service=service, options=options)
driver.get('https://www.scrapingcourse.com/pagination')

pagina_atual = 1
max_paginas = 3

data = []
while pagina_atual <= max_paginas:
    time.sleep(3)
    items = driver.find_elements(By.TAG_NAME,'a')

    for item in items:
        txt = item.text.strip()
        if '$' in txt:
            name, price = txt.split('$',1)
            data.append({
                'pagina':pagina_atual,
                'nome':name.strip(),
                'preço':price
            })

    prox_pagina = driver.find_elements(By.LINK_TEXT, str(pagina_atual+1))
    if prox_pagina:
        prox_pagina[0].click()
        pagina_atual+=1

print(data)

time.sleep(5)
driver.quit()

[{'pagina': 1, 'nome': 'Chaz Kangeroo Hoodie', 'preço': '52'}, {'pagina': 1, 'nome': 'Teton Pullover Hoodie', 'preço': '70'}, {'pagina': 1, 'nome': 'Bruno Compete Hoodie', 'preço': '63'}, {'pagina': 1, 'nome': 'Frankie Sweatshirt', 'preço': '60'}, {'pagina': 1, 'nome': 'Hollister Backyard Sweatshirt', 'preço': '52'}, {'pagina': 1, 'nome': 'Stark Fundamental Hoodie', 'preço': '42'}, {'pagina': 1, 'nome': 'Hero Hoodie', 'preço': '54'}, {'pagina': 1, 'nome': 'Oslo Trek Hoodie', 'preço': '42'}, {'pagina': 1, 'nome': 'Abominable Hoodie', 'preço': '69'}, {'pagina': 1, 'nome': 'Mach Street Sweatshirt', 'preço': '62'}, {'pagina': 1, 'nome': 'Grayson Crewneck Sweatshirt', 'preço': '64'}, {'pagina': 1, 'nome': 'Ajax Full-Zip Sweatshirt', 'preço': '69'}, {'pagina': 2, 'nome': 'Marco Lightweight Active Hoodie', 'preço': '74'}, {'pagina': 2, 'nome': 'Beaumont Summit Kit', 'preço': '42'}, {'pagina': 2, 'nome': 'Hyperion Elements Jacket', 'preço': '51'}, {'pagina': 2, 'nome': 'Montana Wind Jacket', '

In [21]:
import pandas as pd
df = pd.DataFrame(data)
df

,pagina,nome,preço
0,1,Chaz Kangeroo Hoodie,52
1,1,Teton Pullover Hoodie,70
2,1,Bruno Compete Hoodie,63
3,1,Frankie Sweatshirt,60
4,1,Hollister Backyard Sweatshirt,52
5,1,Stark Fundamental Hoodie,42
6,1,Hero Hoodie,54
7,1,Oslo Trek Hoodie,42
8,1,Abominable Hoodie,69
9,1,Mach Street Sweatshirt,62


### Load More (botão “Load more” para carregar mais itens)

In [20]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import ElementNotInteractableException

from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

service = Service(ChromeDriverManager().install())

options = Options()
#options.headlessess = True
driver = webdriver.Chrome(service=service, options=options)
driver.get('https://www.scrapingcourse.com/button-click')

data = []

pagina_atual = 1
for _ in range(100):
    try:
        button = driver.find_element(By.TAG_NAME,'button')
        if button:
            button.click()
            time.sleep(2)
        
            items = driver.find_elements(By.TAG_NAME,'a')
        
            for item in items:
                txt = item.text.strip()
                if '$' in txt:
                    name, price = txt.split('$',1)
                    data.append({
                        'pagina':pagina_atual,
                        'nome':name.strip(),
                        'preço':price
                    })
            pagina_atual += 1
    except ElementNotInteractableException:
        driver.quit()
        break
    
df = pd.DataFrame(data)
df.drop_duplicates(inplace=True)


In [22]:
df

,pagina,nome,preço
0,1,Chaz Kangeroo Hoodie,52
1,1,Teton Pullover Hoodie,70
2,1,Bruno Compete Hoodie,63
3,1,Frankie Sweatshirt,60
4,1,Hollister Backyard Sweatshirt,52
...,...,...,...
1593,15,Leah Yoga Top,39
1594,15,Chloe Compete Tank,39
1595,15,Maya Tunic,29
1596,15,Antonia Racer Tank,34


### Infinite Scrolling (carregar mais itens ao rolar a página)

In [44]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

service = Service(ChromeDriverManager().install())

driver = webdriver.Chrome(service=service, options=options)
driver.get('https://www.scrapingcourse.com/infinite-scrolling')

for _ in range(20):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1)

    items = driver.find_elements(By.TAG_NAME,'a')

    for item in items:
        txt = item.text.strip()
        if '$' in txt:
            name, price = txt.split('$',1)
            data.append({
                'pagina':pagina_atual,
                'nome':name.strip(),
                'preço':price
            })
    
df = pd.DataFrame(data)
df.drop_duplicates(inplace=True)

time.sleep(5)
driver.quit()

In [46]:
df

,pagina,nome,preço
0,4,Chaz Kangeroo Hoodie,52
1,4,Teton Pullover Hoodie,70
2,4,Bruno Compete Hoodie,63
3,4,Frankie Sweatshirt,60
4,4,Hollister Backyard Sweatshirt,52
...,...,...,...
1418,4,Leah Yoga Top,39
1419,4,Chloe Compete Tank,39
1420,4,Maya Tunic,29
1421,4,Antonia Racer Tank,34


### Login 

In [63]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

service = Service(ChromeDriverManager().install())

options = Options()
options.add_argument("--disable-notifications")  # bloqueia pop-ups de notificação
options.add_argument("--disable-extensions")  # desativa extensões
options.add_argument("--disable-popup-blocking")  # desativa bloqueador de pop-ups
options.add_argument("--disable-infobars")  # remove "Chrome is being controlled by automated test software"

EMAIL = 'admin@example.com'
PASSWORD = 'password'

driver = webdriver.Chrome(service=service, options=options)
driver.get('https://www.scrapingcourse.com/login/csrf')

wait = WebDriverWait(driver, 10)

email = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'input[type="email"]')))
password = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'input[type="password"]')))

email.send_keys(EMAIL)
password.send_keys(PASSWORD)

button = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'button')))
button.click()



time.sleep(5)
driver.quit()

### Table Parsing

In [81]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

service = Service(ChromeDriverManager().install())

options = Options()
#options.headlessess = True
driver = webdriver.Chrome(service=service, options=options)
driver.get('https://www.scrapingcourse.com/table-parsing')
time.sleep(5)
tabela = driver.find_element(By.CSS_SELECTOR, 'table')

linhas = tabela.find_elements(By.CSS_SELECTOR, 'tbody tr')
data = []
for linha in linhas:
    cols = [c.text.strip() for c in linha.find_elements(By.TAG_NAME,'td')]
    if cols:
        data.append(cols)

df = pd.DataFrame(data, columns = ['Product ID','Name','Category','Price','In Stock'])

time.sleep(5)
driver.quit()

### Login turnstile